# 08 — Trend Features: Making Tree-Based Models Extrapolate

**Companion notebook to Article 8 of the *Time Series Feature Engineering* series.**

📖 Read the article: [Medium](https://medium.com/@asidd24/trend-features-making-tree-based-models-extrapolate-11af78cc2435) · [LinkedIn Newsletter](https://www.linkedin.com/newsletters/time-series-engineered-7485187061080137728/)

---

> *"Tree-based models cannot extrapolate. If your test set is at a higher trend level than your training set, LightGBM will flatline at the last value it saw. This is not a bug. It is the mathematical property of how decision trees make predictions — and it silently destroys most forecasting models the moment they hit real production data."*

Every article in this series so far has focused on extracting signal from stationary or quasi-stationary dynamics — lag features, rolling windows, seasonal decomposition, and missingness patterns. All of these assume that if you provide a gradient boosted tree with enough informative features, it will forecast accurately.

For strongly trending data, that assumption fails catastrophically. Because decision trees make predictions by partitioning feature space and returning the mean of training targets in each leaf, their predictions are strictly bounded by $[\min(y_{\text{train}}), \max(y_{\text{train}})]$. When demand grows year-over-year or customers scale, tree-based models flatline.

In this notebook, we cover:
1. **The Extrapolation Failure Mode**: Visualizing why LightGBM flatlines on trending data despite rich lag and rolling features.
2. **Feature Class 1 — Linear Time Index (`t_index`)**: The single column that forces trees into the highest-trend leaf.
3. **Feature Class 2 — Polynomial Trend Features**: Modeling curved trajectories, the danger of degree > 2, and explosive test-set error.
4. **Feature Class 3 — Piecewise Trends & Changepoints**: Handling real-world regime shifts and structural breaks with knot features.
5. **The Production Champion — Hybrid Detrend & Refit**: Separating deterministic trend extrapolation from tree-based residual learning.
6. **Leakage Discipline in Walk-Forward CV**: Why trend models must be refit per fold origin.
7. **Comprehensive Benchmark**: Comparing Vanilla LightGBM, `t_index`, Hybrid Detrending, and Linear Models on WMAPE, Bias, and Vandeput Score.


## 1. Setup & Environment

We import standard data science libraries, LightGBM, linear models from scikit-learn, and the reusable modules from `src/`.


In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.linear_model import LinearRegression

from src.data import generate_daily_sales
from src.features import (
    add_lag_features,
    add_rolling_features,
    add_linear_trend,
    add_polynomial_trend,
    add_piecewise_trend,
    detrend_series,
    build_forecasting_feature_matrix,
)
from src.metrics import mae, rmse, wmape, bias, vandeput_score
from src.validation import make_walk_forward_splitter

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 10
np.random.seed(42)


## 2. The Failure Mode Most Tutorials Ignore: Why Trees Can't Extrapolate

Let's generate 2 years (730 days) of daily retail sales with:
- A strong linear upward trend (`trend_slope = 0.15` units/day)
- 7-day weekly seasonality (weekend demand surge)
- Yearly seasonality and Gaussian noise

We split the series chronologically:
- **Train split**: First 510 days (~70%)
- **Test split**: Last 220 days (~30%)


In [ ]:
# 1. Generate trending retail sales
sales = generate_daily_sales(
    n_days=730,
    start_date="2023-01-01",
    trend_slope=0.15,
    weekly_amplitude=15.0,
    yearly_amplitude=20.0,
    noise_std=5.0,
    base_level=100.0,
    seed=42,
)

train_size = 510
train_sales = sales.iloc[:train_size]
test_sales = sales.iloc[train_size:]

print(f"Total series length: {len(sales)} days ({sales.index[0].date()} to {sales.index[-1].date()})")
print(f"Training split:      {len(train_sales)} days ({train_sales.index[0].date()} to {train_sales.index[-1].date()})")
print(f"Test split:          {len(test_sales)} days ({test_sales.index[0].date()} to {test_sales.index[-1].date()})")
print(f"Train target range:  [{train_sales.min():.1f}, {train_sales.max():.1f}]")
print(f"Test target range:   [{test_sales.min():.1f}, {test_sales.max():.1f}]")


Now we build standard, leakage-safe time series features:
- Lags: 1, 7, 14, 28
- Rolling statistics: 7-day and 28-day rolling mean and standard deviation (shifted by 1 day)

We train:
1. **Vanilla LightGBM** (using lag and rolling statistics alone)
2. **Linear Regression** (on the identical features)


In [ ]:
# Build baseline feature matrix (no trend features)
df_features = pd.DataFrame(sales.rename("target"))
df_features = add_lag_features(df_features, "target", lags=[1, 7, 14, 28])
df_features = add_rolling_features(
    df_features, "target", windows=[7, 28], functions=["mean", "std"], min_horizon=1
)
df_features = df_features.dropna()

feature_cols = [c for c in df_features.columns if c != "target"]

# Split feature matrix
X_train = df_features.loc[df_features.index < test_sales.index[0], feature_cols]
y_train = df_features.loc[df_features.index < test_sales.index[0], "target"]

X_test = df_features.loc[df_features.index >= test_sales.index[0], feature_cols]
y_test = df_features.loc[df_features.index >= test_sales.index[0], "target"]

# 1. Fit LightGBM
model_lgb_vanilla = lgb.LGBMRegressor(n_estimators=150, learning_rate=0.05, random_state=42, verbose=-1)
model_lgb_vanilla.fit(X_train, y_train)
pred_lgb_vanilla = pd.Series(model_lgb_vanilla.predict(X_test), index=X_test.index)

# 2. Fit Linear Regression
model_lr = LinearRegression()
model_lr.fit(X_train, y_train)
pred_lr = pd.Series(model_lr.predict(X_test), index=X_test.index)


In [ ]:
# Plot Figure 1: The Extrapolation Failure Mode
plt.figure(figsize=(14, 6))
plt.plot(train_sales.index, train_sales.values, label="Train Actuals", color="#1f77b4", alpha=0.7)
plt.plot(test_sales.index, test_sales.values, label="Test Actuals", color="#1f77b4", lw=2)
plt.plot(pred_lgb_vanilla.index, pred_lgb_vanilla.values, label="LightGBM (Vanilla Lags & Rolling)", color="#d62728", lw=2.2)
plt.plot(pred_lr.index, pred_lr.values, label="Linear Regression (Same Features)", color="#2ca02c", ls="--", lw=2)

plt.axvline(x=test_sales.index[0], color="black", linestyle=":", label="Train/Test Split")
plt.axhline(y=train_sales.max(), color="gray", linestyle="-.", alpha=0.5, label=f"Max Train Target ({train_sales.max():.1f})")

plt.title("Figure 1: The Failure Mode Most Tutorials Ignore — LightGBM Flatlines on Trending Data", fontsize=13, fontweight="bold")
plt.xlabel("Date")
plt.ylabel("Daily Sales Units")
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()


### Diagnostic Insight
Notice the behavior in **Figure 1**:
- **LightGBM (red line)**: Completely flatlines near ~175, precisely bounded by the highest training target values seen in history. It cannot forecast values above the historical ceiling.
- **Linear Regression (green dashed line)**: Captures the slope and continues extrapolating along the growth trajectory.
- In production, this flatline causes severe, systematic under-forecasting and inventory stockouts.


In [ ]:
def compute_all_metrics(actual, forecast, name):
    return {
        "Model": name,
        "MAE": mae(actual, forecast),
        "RMSE": rmse(actual, forecast),
        "WMAPE (%)": wmape(actual, forecast) * 100,
        "Bias": bias(actual, forecast),
        "Vandeput Score": vandeput_score(actual, forecast),
    }

results_comparison = [
    compute_all_metrics(y_test, pred_lgb_vanilla, "Vanilla LightGBM"),
    compute_all_metrics(y_test, pred_lr, "Linear Regression Baseline"),
]

df_results = pd.DataFrame(results_comparison).set_index("Model")
df_results.round(2)


## 3. Feature Class 1 — The Linear Time Index (`t_index`)

The simplest and most effective single trend feature is a monotonically increasing integer counter:
```python
df['t_index'] = np.arange(len(df))
```

### Why Does This Help?
At test time, `t_index` takes values $> N$ (outside the training range). During tree splitting, this forces test instances into the rightmost terminal leaf that contains the highest training target values.

Let's test this using `add_linear_trend`:


In [ ]:
# Add linear trend index
df_linear_trend = df_features.copy()
df_linear_trend = add_linear_trend(df_linear_trend, col_name="t_index")

X_train_lin = df_linear_trend.loc[df_linear_trend.index < test_sales.index[0]].drop(columns=["target"])
y_train_lin = df_linear_trend.loc[df_linear_trend.index < test_sales.index[0], "target"]

X_test_lin = df_linear_trend.loc[df_linear_trend.index >= test_sales.index[0]].drop(columns=["target"])
y_test_lin = df_linear_trend.loc[df_linear_trend.index >= test_sales.index[0], "target"]

# Train LightGBM with t_index
model_lgb_linear = lgb.LGBMRegressor(n_estimators=150, learning_rate=0.05, random_state=42, verbose=-1)
model_lgb_linear.fit(X_train_lin, y_train_lin)
pred_lgb_linear = pd.Series(model_lgb_linear.predict(X_test_lin), index=X_test_lin.index)


In [ ]:
# Plot Figure 3: Impact of adding t_index
plt.figure(figsize=(14, 6))
plt.plot(sales.index, sales.values, label="Actual Sales", color="#1f77b4", alpha=0.5)
plt.plot(pred_lgb_vanilla.index, pred_lgb_vanilla.values, label="LightGBM without t_index (Flatline)", color="#d62728", ls="--", lw=2)
plt.plot(pred_lgb_linear.index, pred_lgb_linear.values, label="LightGBM with t_index (Recovered Trend)", color="#2ca02c", lw=2.2)

plt.axvline(x=test_sales.index[0], color="black", linestyle=":", label="Train/Test Split")
plt.title("Figure 3: One Feature, Massive Lift — Recovering Trend with `t_index`", fontsize=13, fontweight="bold")
plt.xlabel("Date")
plt.ylabel("Daily Sales Units")
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()


### Two Critical Details About `t_index`:
1. **It is a step-plateau recovery, not a continuous mathematical extrapolation**: Because decision trees return step functions, predictions outside the training range plateau at the mean of the highest training leaf. Combined with lag features, this recovers most of the trend, but it will eventually underestimate long-horizon growth.
2. **It assumes linear trajectory**: If the underlying trend curves, accelerates, or breaks, a single linear index cannot capture the geometry. That brings us to Feature Classes 2 and 3.


## 4. Feature Class 2 — Polynomial Trend Features (`t_squared`, `t_cubed`)

When business growth accelerates or decelerates (e.g. exponential adoption or market saturation), a linear counter underfits the curve.

Polynomial trend features extend `t_index` with powers:
- $t_{\text{index}} = t$
- $t_{\text{squared}} = t^2$
- $t_{\text{cubed}} = t^3$

### The Warning on Degree > 2
Higher-degree polynomials fit historical data with high precision, but their extrapolation outside the training domain can be erratic and explosive.

Let's generate a curving sales series and compare polynomial degrees:


In [ ]:
# Generate accelerating non-linear growth
t_steps = np.arange(730)
curved_trend = 100.0 + 0.05 * t_steps + 0.00025 * (t_steps ** 2)
weekly_season = 15.0 * np.sin(2 * np.pi * t_steps / 7)
curved_sales = pd.Series(
    curved_trend + weekly_season + np.random.normal(0, 4.0, 730),
    index=pd.date_range("2023-01-01", periods=730, freq="D"),
    name="sales"
)

train_curved = curved_sales.iloc[:train_size]
test_curved = curved_sales.iloc[train_size:]

# Fit polynomial features at degrees 1, 2, and 5
t_train = np.arange(train_size)
t_test = np.arange(train_size, len(curved_sales))

poly_fits = {}
for deg in [1, 2, 5]:
    poly_coefs = np.polyfit(t_train, train_curved.values, deg=deg)
    poly_fits[deg] = np.polyval(poly_coefs, t_test)


In [ ]:
# Plot Figure 4: Polynomial Extrapolation Trade-off
plt.figure(figsize=(14, 6))
plt.plot(train_curved.index, train_curved.values, label="Train Actuals", color="#1f77b4", alpha=0.7)
plt.plot(test_curved.index, test_curved.values, label="Test Actuals", color="#1f77b4", lw=2)

plt.plot(test_curved.index, poly_fits[1], label="Degree 1 (Linear Underfit)", color="#2ca02c", lw=2)
plt.plot(test_curved.index, poly_fits[2], label="Degree 2 (Quadratic Best Fit)", color="#ff7f0e", lw=2)
plt.plot(test_curved.index, poly_fits[5], label="Degree 5 (Explosive Over-extrapolation)", color="#d62728", ls="--", lw=2)

plt.axvline(x=test_curved.index[0], color="black", linestyle=":", label="Train/Test Split")
plt.title("Figure 4: The Extrapolation Hazard of Higher-Degree Polynomials", fontsize=13, fontweight="bold")
plt.xlabel("Date")
plt.ylabel("Curved Sales Units")
plt.ylim(50, test_curved.max() * 1.5)
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()


> **Production Rule of Thumb**:
> Never exceed degree 2 (`t_squared`) for raw polynomial trend features unless physical or business laws demand it. If you need more flexibility to model changing slopes, use **Piecewise Trends** instead.


## 5. Feature Class 3 — Piecewise Linear Trends & Changepoints

Real-world business time series rarely experience a single uniform growth rate. Instead, growth accelerates with product launches, resets during supply disruptions, and stabilizes during market maturation.

Piecewise trends model this by introducing **knot features** at known changepoint indices:
$$\text{knot}_{k}(t) = \max(0, t - \text{changepoint}_k)$$

Each knot is strictly 0 prior to the changepoint and increases linearly afterward, allowing the model to additively adjust its slope for that regime.


In [ ]:
# Generate a series with two structural regime changes
t = np.arange(730)
regime_trend = np.zeros(730)

# Regime 1 (0 to 250): Moderate slope (+0.08/day)
# Regime 2 (250 to 500): Rapid expansion (+0.35/day)
# Regime 3 (500 to 730): Post-saturation plateau (+0.02/day)
cp1, cp2 = 250, 500
for i in range(730):
    if i < cp1:
        regime_trend[i] = 100 + 0.08 * i
    elif i < cp2:
        regime_trend[i] = 100 + 0.08 * cp1 + 0.35 * (i - cp1)
    else:
        regime_trend[i] = 100 + 0.08 * cp1 + 0.35 * (cp2 - cp1) + 0.02 * (i - cp2)

break_sales = pd.Series(
    regime_trend + 10.0 * np.sin(2 * np.pi * t / 7) + np.random.normal(0, 3, 730),
    index=pd.date_range("2023-01-01", periods=730, freq="D"),
    name="break_sales"
)

# Apply add_piecewise_trend
df_breaks = break_sales.to_frame()
df_breaks = add_piecewise_trend(df_breaks, changepoints=[cp1, cp2])
print("Generated piecewise trend features:")
print(df_breaks.iloc[[100, 300, 600]][["t_index", f"trend_after_{cp1}", f"trend_after_{cp2}"]])


In [ ]:
# Plot Figure 5: Piecewise Regimes & Knots
plt.figure(figsize=(14, 6))
plt.plot(break_sales.index, break_sales.values, label="Sales with Regime Shifts", color="#1f77b4", lw=1.8)
plt.axvline(x=break_sales.index[cp1], color="#d62728", linestyle="--", label=f"Changepoint 1 (Day {cp1})")
plt.axvline(x=break_sales.index[cp2], color="#ff7f0e", linestyle="--", label=f"Changepoint 2 (Day {cp2})")

# Fit single linear trend vs piecewise linear trend
lin_single = LinearRegression().fit(t.reshape(-1, 1), break_sales.values)
plt.plot(break_sales.index, lin_single.predict(t.reshape(-1, 1)), label="Single Linear Fit (Misses Regimes)", color="gray", ls=":")

X_pw = df_breaks[["t_index", f"trend_after_{cp1}", f"trend_after_{cp2}"]]
lin_piecewise = LinearRegression().fit(X_pw, break_sales.values)
plt.plot(break_sales.index, lin_piecewise.predict(X_pw), label="Piecewise Linear Fit (Captures Regimes)", color="#2ca02c", lw=2)

plt.title("Figure 5: Modeling Structural Breaks with Piecewise Changepoints", fontsize=13, fontweight="bold")
plt.xlabel("Date")
plt.ylabel("Sales Units")
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()


### Where Do Changepoints Come From?
1. **Domain Knowledge**: Known promotional launches, price increases, warehouse expansions, regulatory changes.
2. **Algorithmic Detection**: The `ruptures` package (e.g. PELT or BinSeg) or extracting changepoint dates from Facebook Prophet (`model.changepoints`).
3. **Calendar/Fiscal Milestones**: Fiscal year transitions or seasonal resets.


## 6. The Production Champion — The Hybrid "Detrend and Refit" Pattern

The most dependable architecture for strong-trend time series separates trend extrapolation from residual feature learning:

```
Raw Series (Trending)
       │
       ├─► Step 1: Fit Linear Regression on Training portion only
       │
       ▼
Detrend (Target - Trend) ──► Stationary Residuals
                                   │
                                   ▼
Step 2: Train LightGBM with Lags & Rolling Statistics
                                   │
                                   ▼
                            Forecast Residuals
                                   │
                                   ├─► Step 3: Add Extrapolated Linear Trend
                                   ▼
                            Final Forecast (Tracks Trend + Seasonality)
```

Let's test this using `detrend_series`:


In [ ]:
# Step 1: Detrend the training portion (leakage-safe)
detrended_series, trend_line, trend_model = detrend_series(sales, train_end_idx=train_size)

# Step 2: Build features on the detrended stationary series
df_detrended_features = pd.DataFrame(detrended_series.rename("target"))
df_detrended_features = add_lag_features(df_detrended_features, "target", lags=[1, 7, 14, 28])
df_detrended_features = add_rolling_features(
    df_detrended_features, "target", windows=[7, 28], functions=["mean", "std"], min_horizon=1
)
df_detrended_features = df_detrended_features.dropna()

feature_cols_detrend = [c for c in df_detrended_features.columns if c != "target"]

# Split
X_train_det = df_detrended_features.loc[df_detrended_features.index < test_sales.index[0], feature_cols_detrend]
y_train_det = df_detrended_features.loc[df_detrended_features.index < test_sales.index[0], "target"]

X_test_det = df_detrended_features.loc[df_detrended_features.index >= test_sales.index[0], feature_cols_detrend]
y_test_det = df_detrended_features.loc[df_detrended_features.index >= test_sales.index[0], "target"]

# Step 3: Train LightGBM on detrended residuals
model_lgb_detrended = lgb.LGBMRegressor(n_estimators=150, learning_rate=0.05, random_state=42, verbose=-1)
model_lgb_detrended.fit(X_train_det, y_train_det)
pred_detrended_residuals = model_lgb_detrended.predict(X_test_det)

# Step 4: Re-add the linear trend to test predictions
final_forecast_hybrid = pd.Series(
    pred_detrended_residuals + trend_line.loc[X_test_det.index].values,
    index=X_test_det.index,
)


In [ ]:
# Plot Figure 6: 3-Panel Detrend & Refit Architecture
fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)

# Panel 1: Original series + fitted trend line
axes[0].plot(sales.index, sales.values, label="Actual Series", color="#1f77b4", alpha=0.6)
axes[0].plot(trend_line.iloc[:train_size].index, trend_line.iloc[:train_size].values,
             label="Fitted Linear Trend (Train)", color="#2ca02c", lw=2.5)
axes[0].plot(trend_line.iloc[train_size:].index, trend_line.iloc[train_size:].values,
             label="Extrapolated Linear Trend (Test)", color="#2ca02c", ls="--", lw=2.5)
axes[0].axvline(x=test_sales.index[0], color="black", linestyle=":")
axes[0].set_title("Step 1: Fit Linear Trend on Training Data Only & Extrapolate", fontweight="bold")
axes[0].legend(loc="upper left")

# Panel 2: Detrended series (stationary)
axes[1].plot(detrended_series.index, detrended_series.values, label="Detrended Series (Residuals)", color="#9467bd", lw=1.2)
axes[1].axhline(y=0, color="gray", linestyle="--")
axes[1].axvline(x=test_sales.index[0], color="black", linestyle=":")
axes[1].set_title("Step 2: Detrended Target — Stationary with Pure Seasonality & Noise", fontweight="bold")
axes[1].legend(loc="upper left")

# Panel 3: Final Forecast Comparison
axes[2].plot(test_sales.index, test_sales.values, label="Actual Sales", color="#1f77b4", lw=2)
axes[2].plot(pred_lgb_vanilla.index, pred_lgb_vanilla.values, label="Vanilla LightGBM (Flatline)", color="#d62728", ls="--", lw=2)
axes[2].plot(final_forecast_hybrid.index, final_forecast_hybrid.values, label="Hybrid Detrend & Refit (LightGBM + Trend)", color="#2ca02c", lw=2.2)
axes[2].set_title("Step 3 & 4: Final Re-added Forecast Tracks Actuals Continuously", fontweight="bold")
axes[2].legend(loc="upper left")

plt.tight_layout()
plt.show()


### Comprehensive Benchmark Table

Let's compare all 4 strategies on the test split:
1. **Vanilla LightGBM** (standard lags + rolling stats, no trend)
2. **Linear Regression Baseline** (pure linear model)
3. **LightGBM + Linear Time Index (`t_index`)**
4. **Hybrid Detrend & Refit** (LightGBM on residuals + linear trend re-added)


In [ ]:
all_results = [
    compute_all_metrics(y_test, pred_lgb_vanilla, "1. Vanilla LightGBM (No Trend)"),
    compute_all_metrics(y_test, pred_lr, "2. Linear Regression Baseline"),
    compute_all_metrics(y_test_lin, pred_lgb_linear, "3. LightGBM + t_index"),
    compute_all_metrics(y_test, final_forecast_hybrid, "4. Hybrid Detrend & Refit"),
]

df_benchmark = pd.DataFrame(all_results).set_index("Model")
df_benchmark.round(2)


## 7. Strict Leakage Discipline in Walk-Forward Cross-Validation

Trend features present a subtle and dangerous temporal leakage hazard:

### The Three Cardinal Rules:
- **Rule 1 — Fit Linear Trend on Training Data Only**: Never fit a linear regression across the entire time series before splitting. That leaks the future trajectory into training.
- **Rule 2 — Refit Trend at Every Walk-Forward Fold Origin**: In expanding or rolling walk-forward CV, each fold must fit its own independent trend line based solely on data available up to that origin.
- **Rule 3 — Detect Changepoints on Training Windows Only**: If using algorithms like PELT or Prophet to discover structural breaks, run the detection exclusively on the training split.

Let's execute a proper 5-fold walk-forward validation with per-fold trend refitting:


In [ ]:
# 5-fold walk-forward cross-validation with per-fold detrending
splitter = make_walk_forward_splitter(n_splits=5, test_size=35, forecast_horizon=1)
fold_scores_vanilla = []
fold_scores_hybrid = []

for fold, (train_idx, test_idx) in enumerate(splitter.split(sales)):
    # 1. Slice raw data
    y_tr = sales.iloc[train_idx]
    y_te = sales.iloc[test_idx]
    
    # 2. Vanilla LightGBM
    df_fold_vanilla = build_forecasting_feature_matrix(sales.iloc[:test_idx[-1]+1], lags=[1, 7, 14], rolling_windows=[7])
    X_tr_v = df_fold_vanilla.loc[df_fold_vanilla.index.isin(y_tr.index)].drop(columns=["target"])
    y_tr_v = df_fold_vanilla.loc[df_fold_vanilla.index.isin(y_tr.index), "target"]
    X_te_v = df_fold_vanilla.loc[df_fold_vanilla.index.isin(y_te.index)].drop(columns=["target"])
    y_te_v = df_fold_vanilla.loc[df_fold_vanilla.index.isin(y_te.index), "target"]
    
    m_vanilla = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.05, verbose=-1, random_state=42)
    m_vanilla.fit(X_tr_v, y_tr_v)
    p_vanilla = m_vanilla.predict(X_te_v)
    fold_scores_vanilla.append(wmape(y_te_v, p_vanilla) * 100)
    
    # 3. Hybrid Detrend & Refit (refit trend strictly on fold's training index)
    detrended_fold, trend_line_fold, _ = detrend_series(sales.iloc[:test_idx[-1]+1], train_end_idx=len(train_idx))
    df_fold_hybrid = build_forecasting_feature_matrix(detrended_fold, lags=[1, 7, 14], rolling_windows=[7])
    
    X_tr_h = df_fold_hybrid.loc[df_fold_hybrid.index.isin(y_tr.index)].drop(columns=["target"])
    y_tr_h = df_fold_hybrid.loc[df_fold_hybrid.index.isin(y_tr.index), "target"]
    X_te_h = df_fold_hybrid.loc[df_fold_hybrid.index.isin(y_te.index)].drop(columns=["target"])
    
    m_hybrid = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.05, verbose=-1, random_state=42)
    m_hybrid.fit(X_tr_h, y_tr_h)
    p_resid = m_hybrid.predict(X_te_h)
    p_hybrid = p_resid + trend_line_fold.loc[X_te_h.index].values
    fold_scores_hybrid.append(wmape(y_te_v, p_hybrid) * 100)

df_cv = pd.DataFrame({
    "Fold": [f"Fold {i+1}" for i in range(5)],
    "Vanilla LightGBM WMAPE (%)": fold_scores_vanilla,
    "Hybrid Detrending WMAPE (%)": fold_scores_hybrid,
}).set_index("Fold")

df_cv.loc["Average"] = df_cv.mean()
df_cv.round(2)


## 8. Summary & Practical Cheatsheet

| Scenario / Complexity | Recommended Technique | Why It Fits |
| :--- | :--- | :--- |
| **Quick Baseline / Minor Trend** | Simple Linear Time Index (`t_index`) | One line of code; steps predictions to upper training leaf. Recovers 80% of trend at 5% complexity. |
| **Mildly Curved Trend** | Polynomial Degree 2 (`t_squared`) | Captures acceleration without runaway extrapolation error. |
| **Known Structural Breaks / Regimes** | Piecewise Linear Trend (`add_piecewise_trend`) | Models varying slopes per regime using knot features `np.maximum(0, t - cp)`. |
| **Strong Trend in Production (Enterprise Level)** | **Hybrid Detrend & Refit Pattern** | Decouples linear trend extrapolation from tree residual modeling. Unbeatable accuracy and stability. |

---

### Key Takeaways:
1. **Decision trees cannot extrapolate**: Leaf nodes return historical means. On trending time series, unadjusted tree models will flatline.
2. **`t_index` is the pragmatic starting point**: Always include a time counter when working with trending data.
3. **Polynomial degree > 2 is dangerous**: Higher powers cause explosive errors outside the training range.
4. **The Hybrid Detrend & Refit pattern is the production gold standard**: Detrend on train, train LightGBM on stationary residuals, and re-add extrapolated trend at forecast time.
5. **Enforce temporal leakage discipline**: Always refit trend lines and detect changepoints on the training window of each walk-forward fold.

---

### What's Next?
In **Notebook 09 — Seasonality Features: Dummies, Fourier Terms & Beyond**, we tackle the complementary component of time series: explicit seasonal encoding for multi-seasonal, high-frequency, and long-period cycles.
